# 03 — Baseline 1: Default Assignment

---

### What this notebook does

Leaves every order exactly where it is. Each order is filled from its own DC, priority-8 orders
first, then the largest orders by revenue. Nothing is moved.

This is the number every other method has to beat. It is also what a planner gets if nobody
intervenes, so it is the honest "do nothing" comparison.

### What it does not do

No cleaning, no merging, no rule definitions. All of that is in `02` and arrives through
`dom_model`. This notebook only runs the model and writes its answer down.

> The default assignment is `stage_A()` in `dom_model.py`. The greedy in `04` and the MILP in `05`
> both start from the same call, so all three share one starting point.

## 1. Setup

In [7]:
import os, sys, time, json
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

In [9]:
import os, glob, zipfile

# Colab keeps nothing between runtimes, so the shared folder goes on Drive when we can.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/DOM"
except Exception:
    WORK = os.path.abspath("dom_work")

CLEAN   = f"{WORK}/clean"      # notebook 02 writes here
RESULTS = f"{WORK}/results"    # notebooks 03, 04, 05 write here
os.makedirs(CLEAN, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
print("work folder:", WORK)

Mounted at /content/drive
work folder: /content/drive/MyDrive/DOM


In [10]:
# The model lives in exactly one file, written by 02_data_cleaning.
# Data, constraints C1-C7 and the objective all arrive from here.
sys.path.insert(0, WORK)
os.environ["DOM_CLEAN"] = CLEAN
from dom_model import *

print("orders:", len(HEAD), "| focus:", len(FOCUS), "| clean:", len(CLEAN_ORDERS))
print("DCs:", DCS, "| days:", NT)
print("CFG:", CFG)

orders: 1109 | focus: 472 | clean: 637
DCs: [5083, 5385, 5410, 5420, 5490, 5620, 5641, 5773] | days: 32
CFG: {'MIN_FILL_LIFT_PP': 0.05, 'MIN_CASE_LIFT': 100.0, 'FORWARD_COVER_DAYS': 5, 'LEAD_TIME_MILES': 500.0, 'DOCKS_PER_ORDER': 1, 'SAFETY_STOCK_FRAC': 0.0, 'REQUIRE_OBJ_GAIN': True}


## 2. Run the default assignment

`stage_A()` returns two things: the stock left over once everybody has been served from their own
DC, and the per-order record. Only the record matters here — `POOL_A` is what `04` and `05` need.

In [11]:
t0 = time.time()
POOL_A, DEF = stage_A()                     # everyone stays at their own DC
runtime = round(time.time() - t0, 3)

# nothing moves, so every order is flagged as not diverted
BASE = {gf: dict(DEF[gf], diverted=False, chosen_dc=DEF[gf]["dc"], lift=0.0) for gf in DEF}
print(f"{len(BASE)} orders assigned in {runtime}s")

1109 orders assigned in 0.214s


## 3. The numbers

Reported on the 472 focus orders, using the same `metrics()` every other notebook calls. The
objective over all 1,109 orders is shown too, so the 637 clean orders are never dropped.

In [12]:
m = metrics(BASE, "Baseline 1 · default assignment", runtime)
for k, v in m.items():
    print(f"{k:18s} {v:,.4f}" if isinstance(v, float) else f"{k:18s} {v}")

scenario           Baseline 1 · default assignment
objective_focus    44,365,993.7819
objective_all      75,082,939.8596
fill_rate          0.9047
cases_filled       1,311,570.0000
orders_diverted    0
penalty_cost       84,749.3998
shipping_cost      565,479.0000
runtime_s          0.2140


### 3.1 Where the shortfall sits

The focus orders are short by construction. This shows how badly, and which DCs the problem is
concentrated in — useful context for reading the greedy and MILP moves later.

In [13]:
sh = pd.DataFrame([
    dict(order=g, dc=HEAD[g]["default_dc"],
         ordered=HEAD[g]["ordered_cases"], filled=BASE[g]["filled"],
         short=HEAD[g]["ordered_cases"] - BASE[g]["filled"],
         cof=BASE[g]["cof"], penalty=BASE[g]["pen"])
    for g in FOCUS])

print("focus orders fully filled anyway:", int((sh["short"] <= 1e-9).sum()))
print("\nshortfall by default DC:")
print(sh.groupby("dc").agg(orders=("order","count"), ordered=("ordered","sum"),
                           short=("short","sum"), penalty=("penalty","sum"))
        .assign(fill_pct=lambda x: (1 - x["short"]/x["ordered"]) * 100)
        .round(1).to_string())

focus orders fully filled anyway: 24

shortfall by default DC:
      orders   ordered    short  penalty  fill_pct
dc                                                
5083      41   99595.0   9790.0      0.0      90.2
5385      88  310285.0  23723.0  18843.5      92.4
5410      78  210915.0  19627.0  15915.2      90.7
5420      96  343713.0  37938.0  12030.9      89.0
5490      63  160431.0  14135.0  18645.5      91.2
5620      66  260426.0  24627.0  14964.4      90.5
5641      35   53388.0   7549.0   4350.0      85.9
5773       5   11015.0    809.0      0.0      92.7


## 4. Save

Two files, in the format `06_comparison` expects. Every solver notebook writes the same two.

In [14]:
to_frame(BASE, "baseline").to_csv(f"{RESULTS}/results_baseline.csv", index=False)
pd.DataFrame([m]).to_csv(f"{RESULTS}/metrics_baseline.csv", index=False)
print("wrote results_baseline.csv and metrics_baseline.csv")

wrote results_baseline.csv and metrics_baseline.csv


## 5. What this tells us

* The default assignment fills about **90%** of the cases the focus orders asked for, so the gap
  the other two methods are chasing is roughly one tenth of the focus volume.
* Penalty is small next to revenue. That matters: it means a method that chases penalty alone will
  not find much, and the real lever is filling cases.
* Freight here is the cheapest it can be — every order ships from its nearest planned DC. Any
  move in `04` or `05` will make this number worse. That trade is exactly what the objective is
  there to measure.